# Sitemap Link Inventory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/sitemap-link-inventory.ipynb)

**What:** Find every sitemap for a domain (robots.txt, nested indexes, .gz, .txt, common paths) and return all URLs with lastmod, priority, images, hreflang, path groups and duplicates. Optional broken-link, redirect, noindex checks. Partial results never fail the run. $0.001 per sitemap file / link checked.

**Why:** run the [`sitemap-link-inventory`](https://apify.com/transparent_meteorite/sitemap-link-inventory) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/sitemap-link-inventory) for every option.

In [2]:
run_input = {   'startUrls': ['https://crawlee.dev'],
    'maxUrlsPerSite': 10,
    'checkMode': 'sitemapUrls',
    'maxLinksToCheck': 10}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/sitemap-link-inventory').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> Status: RUNNING, Message: 


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:34.322Z ACTOR: Pulling container image of build 0W9H4coMP8KnL1ZLS from registry.


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:34.325Z ACTOR: Creating container.


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:34.396Z ACTOR: Starting container.


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:34.399Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> Status: RUNNING, Message: Sitemap Link Inventory: 10 items so far


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:38.842Z INFO  [Status message]: Done: 11 items


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> 2026-10-13T22:19:38.877Z Thanks for using Sitemap Link Inventory. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/JZZL7210O5lLZZ8b2/issues) with the run URL and we will reply there.


[apify.sitemap-link-inventory runId:pn9pmytAljTMAUwK4] -> Status: SUCCEEDED, Message: Done: 11 items


Run status: SUCCEEDED | items: 11


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["site", "url", "pathGroup", "pathDepth", "priority", "changefreq", "checked", "statusCode", "responseMs"] if c in df.columns]
df[SHOW]

,site,url,pathGroup,pathDepth,priority,changefreq,checked,statusCode,responseMs
0,crawlee.dev,https://crawlee.dev/blog,/blog,1.0,0.5,weekly,True,200.0,274.0
1,crawlee.dev,https://crawlee.dev/blog/archive,/blog,2.0,0.5,weekly,True,200.0,575.0
2,crawlee.dev,https://crawlee.dev/blog/authors,/blog,2.0,0.5,weekly,True,200.0,879.0
3,crawlee.dev,https://crawlee.dev/blog/building-gradcracker-scraper-with-crawlee,/blog,2.0,0.5,weekly,True,200.0,1174.0
4,crawlee.dev,https://crawlee.dev/blog/common-problems-in-web-scraping,/blog,2.0,0.5,weekly,True,200.0,1475.0
5,crawlee.dev,https://crawlee.dev/blog/crawlee-blog-launch,/blog,2.0,0.5,weekly,True,200.0,1479.0
6,crawlee.dev,https://crawlee.dev/blog/crawlee-for-python-v05,/blog,2.0,0.5,weekly,True,200.0,1481.0
7,crawlee.dev,https://crawlee.dev/blog/crawlee-for-python-v06,/blog,2.0,0.5,weekly,True,200.0,1485.0
8,crawlee.dev,https://crawlee.dev/blog/crawlee-for-python-v1,/blog,2.0,0.5,weekly,True,200.0,1481.0
9,crawlee.dev,https://crawlee.dev/blog/crawlee-python-price-tracker,/blog,2.0,0.5,weekly,True,200.0,1477.0


## 5. A quick analysis

In [5]:
df['responseMs']=pd.to_numeric(df['responseMs'],errors='coerce')
print(df['statusCode'].value_counts(dropna=False))
print(df['pathGroup'].value_counts().head(10))
print('Median response (ms):',df['responseMs'].median())

statusCode
200.0    10
NaN       1
Name: count, dtype: int64
pathGroup
/blog    10
Name: count, dtype: int64
Median response (ms): 1476.0


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [sitemap-link-inventory on the Apify Store](https://apify.com/transparent_meteorite/sitemap-link-inventory).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.